# Forêts d'arbres aléatoires : dataset [AMES](http://jse.amstat.org/v19n3/decock.pdf)

## Données

Nous reprenons là où nous nous sommes arrêtés après l'exploration de données : il faut récupérer de nouveau le dataset, et le pré-traiter.

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-ames.git
!ls -l dataset-ames/

In [ ]:
!wc -l dataset-ames/train.csv

In [ ]:
import pandas
import numpy
import matplotlib.pyplot as plt
import seaborn
import sklearn.ensemble
import sklearn.model_selection
import scipy.stats
import warnings

warnings.filterwarnings("ignore")


def preprocess(train_file, test_file):
    train_X = pandas.read_csv(train_file, index_col="Id")
    test_X = pandas.read_csv(test_file, index_col="Id")

    train_y = train_X.pop("SalePrice")

    all_X = pandas.concat([train_X, test_X])

    # Fill with median
    cols_1 = ["LotFrontage"]
    all_X[cols_1] = all_X[cols_1].fillna(train_X[cols_1].median())

    # Fill with mode
    cols_2 = [
        "MSZoning",
        "Electrical",
        "KitchenQual",
        "Exterior1st",
        "Exterior2nd",
        "SaleType",
        "Utilities",
    ]
    all_X[cols_2] = all_X[cols_2].fillna(train_X[cols_2].mode().iloc[0, :])

    # Fill with 0
    cols_4 = [
        "GarageYrBlt",
        "GarageArea",
        "GarageCars",
        "BsmtFinSF1",
        "BsmtFinSF2",
        "BsmtFullBath",
        "BsmtHalfBath",
        "BsmtUnfSF",
        "MasVnrArea",
        "TotalBsmtSF",
    ]
    all_X[cols_4] = all_X[cols_4].fillna(0)

    # Other fills
    cols_5 = ["Functional"]
    all_X[cols_5] = all_X[cols_5].fillna("Typ")

    # On donne à tous les autres NAs la valeur string NA, qui sera une catégorie
    all_X = all_X.fillna("NA")

    # On transforme le codage numérique en string afin que ce soit traité comme
    # une variable catégorielle
    cols_numerical2label = ["MSSubClass"]
    all_X[cols_numerical2label] = all_X[cols_numerical2label].astype(str)

    quality_mapping = dict(NA=0, Po=1, Fa=2, TA=3, Gd=4, Ex=5)
    quality_columns = [
        "BsmtCond",
        "BsmtQual",
        "ExterCond",
        "ExterQual",
        "FireplaceQu",
        "GarageCond",
        "GarageQual",
        "HeatingQC",
        "KitchenQual",
        "PoolQC",
    ]
    street_mapping = dict(NA=0, Grvl=1, Pave=2)
    bsmt_fin_mapping = dict(NA=0, Unf=1, LwQ=2, Rec=3, BLQ=4, ALQ=5, GLQ=6)

    replace_mapping = dict(
        Alley=street_mapping,
        BsmtExposure=dict(NA=0, No=1, Mn=2, Av=3, Gd=4),
        BsmtFinType1=bsmt_fin_mapping,
        BsmtFinType2=bsmt_fin_mapping,
        Functional=dict(Sal=1, Sev=2, Maj2=3, Maj1=4, Mod=5, Min2=6, Min1=7, Typ=8),
        LandSlope=dict(Sev=1, Mod=2, Gtl=3),
        LotShape=dict(IR3=1, IR2=2, IR1=3, Reg=4),
        PavedDrive=dict(NA=0, N=1, P=2, Y=3),
        Street=dict(Grvl=1, Pave=2),
        Utilities=dict(ELO=1, NoSeWa=2, NoSewr=3, AllPub=4),
    )

    for quality_column in quality_columns:
        replace_mapping[quality_column] = quality_mapping

    all_X.replace(replace_mapping, inplace=True)

    print(f"Nombre de NAs : {all_X.isnull().sum().sum()}")

    dummies = pandas.get_dummies(all_X)
    return (
        dummies.iloc[: train_X.shape[0], :],
        train_y,
        dummies.iloc[train_X.shape[0] :, :],
    )

In [ ]:
train_X, train_y, test_X = preprocess("dataset-ames/train.csv", "dataset-ames/test.csv")

Nous avons maintenant prétraité et visualisé les données. Il nous reste à comprendre comment entraîner un modèle, régler ses hyper-paramètres et en tirer les informations qui peuvent nous intéresser.

## Evaluation
Nos données étant maintenant prêtes, il nous faut trouver une méthode d'évaluation correcte pour nos modèles.

*Utilisez [`cross_val_score`](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html) pour écrire une fonction `score` qui évaluera un modèle avec une 5-fold cross-validation qui utilise[`root mean squared error`](https://scikit-learn.org/stable/modules/model_evaluation.html).*

*Attention, la fonction de scoring utilisable est ``neg_root_mean_squared_error`` car dans l'api sklearn, un score à le comportement opposé à une erreur donc pour utiliser une erreur en tant que score, il faut utiliser la version négative de l'erreur. (Une solution avec une erreur élevée doit avoir un score inférieur à une solution avec une erreur faible)*

In [ ]:
# def score(model):
# Votre code ici

### Solution

In [ ]:
def score(model):
    rmse = -sklearn.model_selection.cross_val_score(
        model, train_X, train_y, cv=5, scoring="neg_root_mean_squared_error"
    )
    return rmse.mean(), rmse.std()

## Apprentissage
Nous avons maintenant un outil d'évaluation fiable. Nous pouvons commencer à expérimenter avec des modèles de régression avancés.

*Testez un [`RandomForestRegressor`](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) initialisé avec `n_estimators=10` : ça sera une baseline à surpasser.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(score(sklearn.ensemble.RandomForestRegressor(n_estimators=10)))

## Optimisation
Une des premières choses à faire pour tirer le plus possible d'un modèle est de régler ses paramètres.

*Utilisez [`RandomizedSearchCV`](http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) sur les paramètres `n_estimators`, `max_features`, `max_depth` et `min_samples_leaf` pour trouver une bonne configuration de paramètres.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
# max_features=None correspond à max_features=n_features (cf. doc sklearn)
rscv = sklearn.model_selection.RandomizedSearchCV(
    estimator=sklearn.ensemble.RandomForestRegressor(),
    param_distributions=dict(
        max_samples=[0.6, 0.8, 1.0],
        n_estimators=[10, 100, 500],
        max_features=["log2", "sqrt", None],
        max_depth=[8, 128],
        min_samples_leaf=[3, 5],
    ),
    n_iter=30,
    scoring="neg_root_mean_squared_error",
    cv=2,
    random_state=889,
    n_jobs=-1,
    return_train_score=True,
    verbose=10,
)

rscv.fit(train_X, train_y)

print(f"Meilleur perte obtenue : {-rscv.best_score_}")
print(f"Meilleurs paramètres : {rscv.best_params_}")

# Meilleur modèle entraîné
rfr = rscv.best_estimator_

## Utilisation du modèle pour la prédiction

On peut d'ores et déjà faire des prédictions avec notre meilleur modèle entraîné.

*Calculez les prédictions du modèle sur l'ensemble de test grâce à la fonction [`sklearn.ensemble.RandomForestRegressor.predict`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html#sklearn.ensemble.RandomForestRegressor.predict)*.

In [ ]:
# Votre code ici

### Solution

In [ ]:
predictions = rfr.predict(test_X)
print(f"Taille du test set : {test_X.shape}")
print(f"Taille des prédictions : {predictions.shape}")

seaborn.distplot(predictions)
plt.title("Densité des prédictions")
plt.xlabel("Prix en $")
plt.ylabel("Densité")
plt.show()

## Affichage des features importantes

Ce modèle étant satisfaisant pour l'instant, intéressons-nous aux possibilités qu'offre Random Forest. Il est par exemple possible de calculer l'importance des features en observant la chute moyenne en erreur (ou impureté) qu'elles entraînent.

*Utilisez l'attribut `feature_importances_` de votre Random Forest pour comprendre quelles sont les features les plus importantes.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
importances = pandas.DataFrame(
    {"importance": rfr.feature_importances_}, index=train_X.columns
)
importances.sort_values(by="importance", ascending=False, inplace=True)
importances.iloc[:20, :][::-1].plot.barh()
plt.show()

## Performances de généralisation
Il est aussi possible de mesurer la généralisation d'une forêt sur les exemples que ses arbres n'ont pas pu voir de par le processus de boostrapping.

*Utilisez l'attribut `oob_score_` pour mesurer l'impact du nombre d'arbres dans la forêt. Pour cela, entraînez un modèle avec 1 à 20 arbres et reportez le score OOB dans un plot. L'OOB score que calcule `scikit-learn` est le score R². De bonnes ressources pour l'interpréter sont disponibles sur stats.stackexchange : [ici](https://stats.stackexchange.com/questions/70704/interpreting-out-of-bag-error-estimate-for-randomforestregressor) et [là](https://stats.stackexchange.com/questions/133406/is-a-negative-oob-score-possible-with-scikit-learns-randomforestregressor).*

In [ ]:
# Votre code ici

### Solution

In [ ]:
oob_scores = []
max_trees = 50
for i in range(1, max_trees + 1):
    rfr.set_params(n_estimators=i, oob_score=True)
    rfr.fit(train_X, train_y)
    # Enregistrement du score OOB pour chaque nombre d'arbres
    # score OOB = score R².
    oob_score = rfr.oob_score_
    oob_scores.append(oob_score)

plt.title("Effet du nombre d'arbres sur la performance de la forêt")
plt.xlabel("Nombre d'arbres")
plt.ylabel("Score OOB")
plt.plot(range(1, max_trees + 1), oob_scores)
plt.show()

In [ ]:
# Zoom sur les scores OOB quand ils deviennent positifs (modèles meilleurs qu'un
# prédicteur constant)
oob_array = numpy.array(oob_scores)
left_context = 3
first_positive = numpy.where(oob_array > 0)[0][0]
first_point = max(1, first_positive - left_context)
plt.title("Effet du nombre d'arbres sur la performance de la forêt")
plt.xlabel("Nombre d'arbres")
plt.ylabel("Score OOB")
plt.plot(range(first_point + 1, max_trees + 1), oob_array[first_point:])

In [ ]:
# Affichage du premier arbre de la foret obtenu à l'aide de GraphViz
from sklearn.tree import export_graphviz

# Export as dot file
export_graphviz(
    rfr.estimators_[0],
    out_file="tree.dot",
    feature_names=train_X.columns,
    rounded=True,
    proportion=False,
    precision=2,
    filled=True,
)

# Convert to png using system command (requires Graphviz)
from subprocess import call

call(["dot", "-Tpng", "tree.dot", "-o", "tree.png", "-Gdpi=600"])
# Convert to svg (plus facile à visualiser dans un navigateur après l'avoir téléchargé)
call(["dot", "-Tsvg", "tree.dot", "-o", "tree.svg"])

# Display in jupyter notebook
from IPython.display import Image

Image(filename="tree.png")